In [38]:
!pip install rocketpy


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd
from rocketpy import Environment, SolidMotor, Rocket, Flight
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

NUM_SIMULATIONS = 100
ENV_LAT, ENV_LON, ENV_ELEVATION = 0.0, 0.0, 0.0

missile_types = ["Khalij_Fars", "Qiam_1", "Zolfaghar", "Shahab_3"]
data_dict = {m_type: [] for m_type in missile_types}
sim_id_tracker = {m_type: 1 for m_type in missile_types}

# [초고고도 외삽 대기 함수] 무한한 고도에서도 터지지 않는 지수 감소 기압/온도 모델
def custom_pressure(z):
    z = np.asarray(z)
    return 101325.0 * np.exp(-z / 8420.0)

def custom_temperature(z):
    z = np.asarray(z)
    return np.where(z < 11000.0, 288.15 - 0.0065 * z, 216.65)

def create_missile(missile_type, mass_variation, drag_variation):
    # 실제 군사 1:1 제원 세팅 (Fact Base)
    if missile_type == 'Khalij_Fars':
        radius = 0.61 / 2
        propellant_mass = 2200 * mass_variation  # 연료 2.2톤
        mass_without_motor = 1120 * mass_variation # 본체 및 탄두 1.1톤 (총 3.3톤)
        burn_time = 25.0
        thrust_curve = [(0, 200000), (burn_time, 200000)] # 추력 약 20톤(200kN)
        
    elif missile_type == 'Qiam_1':
        radius = 0.88 / 2
        propellant_mass = 4500 * mass_variation  # 연료 4.5톤
        mass_without_motor = 1655 * mass_variation # (총 6.1톤)
        burn_time = 50.0
        thrust_curve = [(0, 250000), (burn_time, 250000)]
        
    elif missile_type == 'Zolfaghar':
        radius = 0.68 / 2
        propellant_mass = 3500 * mass_variation  # 연료 3.5톤
        mass_without_motor = 1120 * mass_variation # (총 4.6톤)
        burn_time = 40.0
        thrust_curve = [(0, 220000), (burn_time, 220000)]
        
    elif missile_type == 'Shahab_3':
        radius = 1.25 / 2
        propellant_mass = 12000 * mass_variation # 연료 12톤
        mass_without_motor = 3000 * mass_variation # (총 15톤)
        burn_time = 70.0
        thrust_curve = [(0, 450000), (burn_time, 450000)] # 추력 약 45톤(450kN)

    # 지정된 연료 질량(propellant_mass)에 맞춰 연료통 높이(grain_height) 자동 역산 계산
    grain_inner = radius * 0.2
    grain_outer = radius * 0.9
    grain_density = 1800
    area = np.pi * (grain_outer**2 - grain_inner**2)
    grain_height = propellant_mass / (grain_density * area)
    
    applied_drag = 0.3 * drag_variation
        
    motor = SolidMotor(
        thrust_source=thrust_curve,
        dry_mass=200.0, # 빈 모터통 무게
        dry_inertia=(50, 50, 10),
        nozzle_radius=radius * 0.8,
        grain_number=1,
        grain_density=grain_density,
        grain_outer_radius=grain_outer,
        grain_initial_inner_radius=grain_inner,
        grain_initial_height=grain_height,
        grain_separation=0.0,
        grains_center_of_mass_position=grain_height/2,
        center_of_dry_mass_position=1.0,
        coordinate_system_orientation="nozzle_to_combustion_chamber"
    )
    
    missile = Rocket(
        radius=radius,
        mass=mass_without_motor,
        inertia=(2000, 2000, 100),
        power_off_drag=applied_drag, 
        power_on_drag=applied_drag,
        center_of_mass_without_motor=2.0,
        coordinate_system_orientation="tail_to_nose"
    )
    missile.add_motor(motor, position=0)
    missile.add_nose(length=radius*3, kind="vonKarman", position=3.0)
    missile.add_trapezoidal_fins(n=4, span=radius*1.5, root_chord=radius*1.5, tip_chord=radius*0.5, position=-1.0)
    
    return missile, burn_time, applied_drag

print("초고고도 외삽 대기 모델 기반 1:1 실제 제원 시뮬레이션을 시작합니다...")

for i in tqdm(range(NUM_SIMULATIONS)):
    try:
        missile_type = np.random.choice(missile_types)
        elevation = np.random.uniform(60, 80) 
        azimuth = np.random.uniform(0, 360)
        mass_var = np.random.uniform(0.95, 1.05)   
        drag_var = np.random.uniform(0.90, 1.10)   
        
        wind_u_val = np.random.normal(0, 10.0) 
        wind_v_val = np.random.normal(0, 10.0) 
        
        env = Environment(latitude=ENV_LAT, longitude=ENV_LON, elevation=ENV_ELEVATION)
        
        # [핵심] 고도 80km 한계를 없애는 커스텀 압력/온도 모델 주입
        env.set_atmospheric_model(
            type='custom_atmosphere', 
            pressure=custom_pressure, 
            temperature=custom_temperature,
            wind_u=wind_u_val,
            wind_v=wind_v_val
        )
        
        missile, burn_time, applied_drag = create_missile(missile_type, mass_var, drag_var)
        
        # 실제 비행 시간은 15분에 달하므로 최대 1200초로 여유 설정, 연산 최적화를 위해 max_time_step 0.5초 적용
        flight = Flight(rocket=missile, environment=env, rail_length=10.0, 
                        inclination=elevation, heading=azimuth, max_time=1200, max_time_step=0.5)
        
        target_x = flight.x(flight.t_final)
        target_y = flight.y(flight.t_final)
        target_t = flight.t_final
        
        # 비정상 추락 필터링
        if target_t < 60.0:
            continue
            
        current_sim_id = sim_id_tracker[missile_type]
        sim_id_tracker[missile_type] += 1
        target_list = data_dict[missile_type]
        
        # 1초 단위 샘플링 (1만 개 * 600초 비행 = 엄청난 데이터양이므로 1초 단위가 적합)
        time_steps = np.arange(1.0, target_t, 1.0)
        time_steps = np.append(time_steps, target_t)
        
        initial_propellant_mass = missile.motor.propellant_initial_mass
        total_dry_mass = missile.mass + missile.motor.dry_mass
        
        for t in time_steps:
            if t < burn_time:
                current_mass = total_dry_mass + initial_propellant_mass * (1 - (t / burn_time))
            else:
                current_mass = total_dry_mass
                
            vx, vy, vz = flight.vx(t), flight.vy(t), flight.vz(t)
            v_mag = np.sqrt(vx**2 + vy**2 + vz**2)
            current_z = flight.z(t)
            
            rho = 1.225 * np.exp(-current_z / 8420.0) 
            mach_val = v_mag / 340.3
            dyn_pres = 0.5 * rho * (v_mag ** 2)
            
            try: pitch_val = float(flight.inclination(t))
            except: pitch_val = np.degrees(np.arcsin(vz / (v_mag + 1e-6)))
            
            try: yaw_val = float(flight.heading(t))
            except: yaw_val = np.degrees(np.arctan2(vy, vx))

            target_list.append({
                'sim_id': current_sim_id,
                'time_step': round(t, 2),
                'MC_mass_var': round(mass_var, 3),
                'MC_drag_coeff': round(applied_drag, 3),
                'MC_wind_u': round(wind_u_val, 2),
                'MC_wind_v': round(wind_v_val, 2),
                'x': flight.x(t), 'y': flight.y(t), 'z': flight.z(t),
                'vx': vx, 'vy': vy, 'vz': vz,
                'ax': flight.ax(t), 'ay': flight.ay(t), 'az': flight.az(t),
                'pitch': pitch_val, 'yaw': yaw_val,
                'mach': mach_val, 'dynamic_pressure': dyn_pres, 'air_density': rho,
                'current_mass': round(current_mass, 2),
                'Target_X': target_x, 'Target_Y': target_y, 'Target_T': target_t
            })
            
    except Exception:
        continue

print("\n✅ 실제 스케일 시뮬레이션 완료 및 파일 저장 중...")
for m_type in missile_types:
    if len(data_dict[m_type]) > 0:
        df_temp = pd.DataFrame(data_dict[m_type])
        filename = f"{m_type}_RealScale_Trajectory.csv"
        df_temp.to_csv(filename, index=False)
        print(f"- {m_type}: {sim_id_tracker[m_type] - 1}건 궤적 저장 완료 ({filename})")

초고고도 외삽 대기 모델 기반 1:1 실제 제원 시뮬레이션을 시작합니다...


  0%|          | 9/10000 [00:06<1:56:49,  1.43it/s]capi_return is NULL
Call-back cb_f_in_lsoda__user__routines failed.



KeyboardInterrupt: 